In [4]:
import importlib
import sys
from pathlib import Path

import pandas as pd
import plotly.io as pio
from IPython.display import display

# Interactive Plotly charts in Jupyter / VS Code / Cursor
pio.renderers.default = "plotly_mimetype+notebook_connected"

sys.path.insert(0, str(Path("..").resolve()))
sys.path.insert(0, str(Path(".").resolve()))

import plotter

importlib.reload(plotter)
from plotter import Scale, plot_research_price, plot_series

from engine import EntryType, latest_research_entries, steps_path

OUTCOMES_DIR = Path("../outcomes")
RESEARCH_NAME = "Investing at ATH"


def load_research(research: str = RESEARCH_NAME) -> dict[str, pd.DataFrame]:
    """Load steps for every experiment in the latest batch of `research`."""
    entries = latest_research_entries(OUTCOMES_DIR, research)
    outcomes = {}
    for entry in entries:
        key = entry.get("name") or entry["id"]
        outcomes[key] = pd.read_json(steps_path(OUTCOMES_DIR, entry), lines=True)
    return outcomes


outcomes = load_research()

# for name, steps in outcomes.items():
#     print(name)
#     display(steps)

plot_series(
    outcomes,
    "equity",
    journal=[
        # EntryType.DEPOSIT,
            EntryType.ORDER_FILLED,
            EntryType.INTEREST,
            # EntryType.IS_ATH
            ],
)


In [5]:
TICKER = "SPX"
# One shared price path; journal markers from every experiment in the research
plot_research_price(
    outcomes,
    TICKER,
    bands=("high", "low"),
    journal=[
        #EntryType.DEPOSIT,
        # EntryType.ORDER_FILLED,
        # EntryType.INTEREST,
        # EntryType.ORDER_CANCELLED,
        EntryType.IS_ATH,
    ],
)


In [6]:
cash_frames = {}
for name, steps in outcomes.items():
    df = steps.copy()
    df["available_usd"] = df["balances"].apply(lambda b: float(b["USD"]))
    df["frozen_usd"] = df["frozen_usd"].fillna(0).astype(float)
    df["total_usd"] = df["available_usd"] + df["frozen_usd"]
    cash_frames[name] = df

plot_series(cash_frames, "total_usd", scale=Scale.LINEAR)
plot_series(cash_frames, "available_usd", scale=Scale.LINEAR)
plot_series(cash_frames, "frozen_usd", scale=Scale.LINEAR)
